## Test notebook

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import optuna
import xgboost as xgb
import lightgbm as lgb
import scipy.stats as stats
from sklearn.model_selection import GroupKFold # determine what type of CV split to do
from sklearn.metrics import mean_absolute_error
from scipy.ndimage import binary_erosion
from skimage.morphology import local_maxima
from skimage.measure import label
from PIL import Image
from pathlib import Path
from lightgbm import early_stopping
%matplotlib inline

c:\Users\eddyk\miniconda3\envs\staix\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load all data.

In [ ]:
from src.data_loader import (
    find_data_dir,
    load_train_data,
    load_val_data,
    load_train_pngs,
    load_val_pngs
)

# Get data directory
root = find_data_dir()

# Get training data and validation covariates
train_all_drugs, train_all_opioids, train_all_stims = load_train_data(root)
val_cov = load_val_data(root)

# Get image data
train_imgs, train_img_names = load_train_pngs(root) 
val_imgs, val_img_names = load_val_pngs(root)

Feature engineering pipeline.

In [3]:
from src.features import create_all_features

# Run feature engineering pipeline on train and val
train_all_drugs = create_all_features(
    train_all_drugs, 
    train_imgs, 
    train_img_names
)
train_all_opioids = create_all_features(
    train_all_opioids,
    train_imgs,
    train_img_names
)
train_all_stims = create_all_features(
    train_all_stims,
    train_imgs,
    train_img_names
)
X_val = create_all_features(
    val_cov,
    val_imgs,
    val_img_names
)

Extract covariates and prediction target.

In [ ]:
from src.tuning import return_covs_and_target

X_train_drugs, y_train_drugs = return_covs_and_target(train_all_drugs)
X_train_opioids, y_train_opioids = return_covs_and_target(train_all_opioids)
X_train_stims, y_train_stims = return_covs_and_target(train_all_stims)

Hyperparameter tuning and model selection.

In [ ]:
from src.tuning import find_best_regressor

# Run experiments to find optimal regressor for each dataset
drug_results = find_best_regressor(
    X_train_drugs, 
    y_train_drugs,
    num_folds = 10
)
opioid_results = find_best_regressor(
    X_train_opioids, 
    y_train_opioids,
    num_folds = 10
)
stim_results = find_best_regressor(
    X_train_stims, 
    y_train_stims,
    num_folds = 10
)

Retrain with optimal hyperparameters.

Run models on validation and write to csv.